# TabFM no BigQuery: detecção de fraude *zero-shot* com `AI.PREDICT`

**Demo para Colab Enterprise** · BigQuery ML · TabFM (Preview)

---

## O que é o TabFM?

**TabFM** é um *foundation model* para dados tabulares desenvolvido pelo Google Research. Assim como um LLM aprende uma tarefa a partir de exemplos no prompt (*in-context learning*), o TabFM **lê sua tabela histórica como exemplos em contexto e gera previsões para a tabela alvo em um único passo**, sem treinar, ajustar ou implantar modelo algum.

No BigQuery ele é exposto por duas funções SQL nativas:

| Função | Para quê |
|---|---|
| `AI.PREDICT(treino, predição, label_col => '...')` | Classificação ou regressão *zero-shot* |
| `AI.EVALUATE(treino, teste, label_col => '...')` | Métricas de qualidade (precision/recall/F1 ou MAE/R²) |

A tarefa (classificação × regressão) é **inferida pelo tipo da coluna de rótulo**: `BOOL`/`STRING` → classificação; `INT64`/`FLOAT64`/`NUMERIC` → regressão.

## O que vamos fazer neste notebook

1. Gerar um dataset sintético realista de **transações de cartão** (~120 mil linhas, ~3 % de fraude) e carregá-lo no BigQuery.
2. Rodar a **primeira previsão de fraude com uma única query SQL**, sem `CREATE MODEL`.
3. Avaliar qualidade com `AI.EVALUATE` e com análises de negócio (matriz de confusão, curva precision-recall, capacidade de alertas).
4. Mostrar **adaptação a um novo padrão de fraude sem re-treino**, só mudando a janela do `WHERE`.
5. Comparar com um modelo clássico do BigQuery ML (`BOOSTED_TREE_CLASSIFIER`) em esforço, tempo e qualidade.
6. Mostrar regressão com a mesma sintaxe e discutir operacionalização, custos e limites.

> **Pré-requisitos:** projeto GCP com BigQuery habilitado, permissão para criar datasets/tabelas e executar queries (`roles/bigquery.user` + `roles/bigquery.dataEditor` no dataset). O TabFM está em **Preview**; suporte durante o preview via `bqml-feedback@google.com`.
>
> **Tempo estimado:** ~25 min ponta a ponta. Cada chamada de `AI.PREDICT` sobre ~100 mil linhas de treino leva alguns minutos; aproveite o tempo para explicar o conceito.

**Referências**
- Documentação: [`AI.PREDICT`](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-predict) · [`AI.EVALUATE`](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-evaluate)
- Blog: [Introducing TabFM in BigQuery](https://cloud.google.com/blog/products/data-analytics/tabfm-adds-predictive-ml-to-bigquery)
- Modelo: [TabFM: a zero-shot foundation model for tabular data](https://research.google/blog/introducing-tabfm-a-zero-shot-foundation-model-for-tabular-data/)

## 1. Configuração

Ajuste `PROJECT_ID`. O dataset será criado automaticamente na região indicada.

In [ ]:
# @title Parâmetros da demo { display-mode: "form" }
PROJECT_ID = "seu-projeto-gcp"  # @param {type:"string"}
DATASET    = "tabfm_fraud_demo"  # @param {type:"string"}
LOCATION   = "US"                # @param {type:"string"}

N_ROWS     = 120000   # @param {type:"integer"}
FRAUD_RATE = 0.025    # @param {type:"number"}
SEED       = 42       # @param {type:"integer"}

FQ = f"{PROJECT_ID}.{DATASET}"          # prefixo totalmente qualificado
T  = f"`{FQ}.transactions`"              # tabela principal
print(f"Projeto: {PROJECT_ID} | Dataset: {FQ} | Região: {LOCATION}")

In [ ]:
import time, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

# No Colab (não Enterprise) descomente para autenticar:
# from google.colab import auth; auth.authenticate_user()

bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Paleta e estilo dos gráficos
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8985"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold"})

def run_sql(sql: str, show: bool = True, max_rows: int = 10) -> pd.DataFrame:
    """Executa uma query, imprime tempo/bytes e devolve um DataFrame."""
    sql = textwrap.dedent(sql)
    t0 = time.time()
    job = bq.query(sql)
    df = job.result().to_dataframe()
    elapsed = time.time() - t0
    mb = (job.total_bytes_processed or 0) / 1e6
    print(f"⏱ {elapsed:,.1f}s · {mb:,.1f} MB processados · job_id={job.job_id}")
    if show and len(df):
        display(df.head(max_rows))
    return df

ds = bigquery.Dataset(FQ); ds.location = LOCATION
bq.create_dataset(ds, exists_ok=True)
print("Dataset pronto:", FQ)

## 2. Dados sintéticos: transações de cartão com fraude

Não usamos dados reais de cliente. O gerador abaixo cria um cenário **estatisticamente plausível** para uma instituição financeira:

- **~120 mil transações** de jan a jul/2026, **20 mil clientes**, **taxa de fraude ≈ 2,5 – 3 %** (forte desbalanceamento, como na vida real).
- Fraude tem comportamento característico mas **não determinístico**: madrugada, e-commerce/tarja, categorias de alto risco (eletrônicos, viagem, games, cripto), valores bimodais (*teste de cartão* de poucos reais ou *esvaziar limite*), dispositivo novo, alta velocidade de transações, muitas falhas de autenticação, contas novas.
- **30 % das fraudes são "camufladas"**: copiam o perfil de uma transação legítima e só deixam sinais fracos. **3 % das legítimas parecem suspeitas** (viagem internacional, compra alta). Isso evita um problema trivial.
- **Um padrão de fraude novo surge em 01/06/2026** ("ataque por aproximação em games digitais": micro-pagamentos repetidos, cartão presente, perto de casa). Ele não existe no histórico anterior; usaremos isso para mostrar adaptação sem re-treino.

### Features (18, dentro do limite de 20 do TabFM)

| Coluna | Tipo | Descrição |
|---|---|---|
| `amount` | FLOAT64 | valor da transação (R$) |
| `merchant_category` | STRING | categoria do estabelecimento (10 valores) |
| `channel` | STRING | `pos_chip`, `pos_tarja`, `online`, `aproximacao`, `caixa_eletronico` |
| `card_present` | BOOL | cartão fisicamente presente |
| `is_international` | BOOL | transação internacional |
| `distance_from_home_km` | FLOAT64 | distância do endereço do cliente |
| `new_device` | BOOL | dispositivo nunca visto (canal online) |
| `txn_count_24h` | INT64 | velocidade: transações nas últimas 24 h |
| `distinct_merchants_24h` | INT64 | lojistas distintos em 24 h |
| `failed_auth_24h` | INT64 | falhas de autenticação em 24 h |
| `customer_age` | INT64 | idade do cliente |
| `account_age_days` | INT64 | idade da conta em dias |
| `avg_amount_30d` | FLOAT64 | ticket médio do cliente em 30 dias |
| `amount_to_avg_ratio` | FLOAT64 | `amount / avg_amount_30d` |
| `credit_limit_utilization` | FLOAT64 | utilização do limite (0-1) |
| `prior_chargebacks` | INT64 | chargebacks anteriores do cliente |
| `hour_of_day`, `day_of_week` | INT64 | hora e dia da semana |

Colunas de **identificação/tempo** (`transaction_id`, `customer_id`, `transaction_ts`) **não** são features: serão excluídas da tabela de treino com `SELECT * EXCEPT(...)`. O rótulo é `is_fraud` (`BOOL`).

In [ ]:
MERCHANT_CATEGORIES = [
    "supermercado", "restaurante", "combustivel", "farmacia", "vestuario",
    "servicos_publicos", "eletronicos", "viagem", "games_digitais", "cripto",
]
# Probabilidade de cada categoria em transações legítimas vs. fraudulentas
LEGIT_MCC_P = np.array([0.22, 0.18, 0.12, 0.08, 0.10, 0.08, 0.08, 0.06, 0.05, 0.03])
FRAUD_MCC_P = np.array([0.04, 0.04, 0.03, 0.02, 0.06, 0.02, 0.28, 0.16, 0.18, 0.17])

CHANNELS = ["pos_chip", "pos_tarja", "online", "aproximacao", "caixa_eletronico"]
LEGIT_CH_P = np.array([0.30, 0.05, 0.35, 0.25, 0.05])
FRAUD_CH_P = np.array([0.05, 0.22, 0.58, 0.10, 0.05])

START = pd.Timestamp("2026-01-01")
END = pd.Timestamp("2026-08-01")  # exclusivo


def _hours(rng: np.random.Generator, n: int, fraud: bool) -> np.ndarray:
    if fraud:
        # fraude concentrada de madrugada
        mix = rng.random(n) < 0.55
        night = rng.integers(0, 6, n)
        day = rng.integers(6, 24, n)
        return np.where(mix, night, day)
    # legítimo: pico comercial e noite
    return np.clip(rng.normal(14, 5, n).round().astype(int), 0, 23)


def generate_transactions(n_rows: int = 120_000, fraud_rate: float = 0.025,
                          seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n_fraud = int(n_rows * fraud_rate)
    n_legit = n_rows - n_fraud

    # ---------- população de clientes ----------
    n_customers = 20_000
    cust_age = np.clip(rng.normal(42, 14, n_customers).round(), 18, 85).astype(int)
    cust_acct_age = np.clip(rng.exponential(900, n_customers).round(), 5, 6000).astype(int)
    cust_avg_amt = np.clip(rng.lognormal(4.3, 0.6, n_customers), 20, 3000)
    cust_limit = np.clip(cust_avg_amt * rng.uniform(8, 40, n_customers), 500, 60000).round(-2)
    cust_prior_cb = rng.choice([0, 0, 0, 0, 0, 0, 0, 0, 1, 2], n_customers)

    def build(n: int, fraud: bool) -> pd.DataFrame:
        # fraudadores atacam preferencialmente contas novas e com histórico ruim
        if fraud:
            w = 1.0 / (cust_acct_age + 60.0) * (1 + cust_prior_cb)
            w /= w.sum()
            cid = rng.choice(n_customers, n, p=w)
        else:
            cid = rng.integers(0, n_customers, n)

        ts = START + pd.to_timedelta(rng.uniform(0, (END - START).total_seconds(), n), unit="s")
        hour = _hours(rng, n, fraud)
        ts = ts.normalize() + pd.to_timedelta(hour, unit="h") + pd.to_timedelta(rng.integers(0, 3600, n), unit="s")

        mcc = rng.choice(MERCHANT_CATEGORIES, n, p=FRAUD_MCC_P if fraud else LEGIT_MCC_P)
        ch = rng.choice(CHANNELS, n, p=FRAUD_CH_P if fraud else LEGIT_CH_P)
        card_present = np.isin(ch, ["pos_chip", "pos_tarja", "aproximacao", "caixa_eletronico"])

        avg30 = cust_avg_amt[cid] * rng.lognormal(0, 0.15, n)
        if fraud:
            # bimodal: "teste de cartão" (micro) ou "esvaziar limite" (alto)
            micro = rng.random(n) < 0.25
            amt = np.where(micro,
                           rng.uniform(1, 15, n),
                           avg30 * rng.lognormal(1.4, 0.7, n))
        else:
            amt = avg30 * rng.lognormal(0, 0.55, n)
        amt = np.clip(amt, 1, 50000).round(2)

        if fraud:
            intl = rng.random(n) < 0.35
            dist = np.where(intl, rng.uniform(1500, 12000, n), rng.exponential(120, n))
            new_dev = (rng.random(n) < 0.70) & ~card_present
            txn24 = rng.poisson(5.5, n) + 1
            merch24 = np.minimum(txn24, rng.poisson(3.5, n) + 1)
            failed = rng.poisson(1.6, n)
            util = np.clip(rng.beta(4, 3, n), 0, 1)
        else:
            intl = rng.random(n) < 0.03
            dist = np.where(intl, rng.uniform(1500, 12000, n), rng.exponential(12, n))
            new_dev = (rng.random(n) < 0.08) & ~card_present
            txn24 = rng.poisson(1.6, n) + 1
            merch24 = np.minimum(txn24, rng.poisson(1.2, n) + 1)
            failed = rng.poisson(0.12, n)
            util = np.clip(rng.beta(2, 5, n), 0, 1)

        df = pd.DataFrame({
            "customer_id": cid,
            "transaction_ts": ts,
            "amount": amt,
            "merchant_category": mcc,
            "channel": ch,
            "card_present": card_present,
            "is_international": intl,
            "distance_from_home_km": dist.round(1),
            "new_device": new_dev,
            "txn_count_24h": txn24,
            "distinct_merchants_24h": merch24,
            "failed_auth_24h": failed,
            "customer_age": cust_age[cid],
            "account_age_days": cust_acct_age[cid],
            "avg_amount_30d": avg30.round(2),
            "credit_limit_utilization": util.round(3),
            "prior_chargebacks": cust_prior_cb[cid],
            "is_fraud": fraud,
        })
        return df

    legit = build(n_legit, fraud=False)
    fraud = build(n_fraud, fraud=True)

    # ---------- "fraude difícil": 30% das fraudes camufladas como legítimas ----------
    n_hard = int(len(fraud) * 0.30)
    hard_idx = rng.choice(len(fraud), n_hard, replace=False)
    camo = build(n_hard, fraud=False)
    for col in ["amount", "merchant_category", "channel", "card_present", "distance_from_home_km",
                "is_international", "txn_count_24h", "distinct_merchants_24h", "failed_auth_24h",
                "credit_limit_utilization"]:
        fraud.loc[hard_idx, col] = camo[col].values
    # mantém sinal fraco: dispositivo novo e horário
    fraud.loc[hard_idx, "new_device"] = (rng.random(n_hard) < 0.45) & ~fraud.loc[hard_idx, "card_present"].values

    # ---------- ruído em legítimas: 3% parecem suspeitas (viagens, compras grandes) ----------
    n_noisy = int(len(legit) * 0.03)
    noisy_idx = rng.choice(len(legit), n_noisy, replace=False)
    legit.loc[noisy_idx, "is_international"] = True
    legit.loc[noisy_idx, "distance_from_home_km"] = rng.uniform(1500, 9000, n_noisy).round(1)
    legit.loc[noisy_idx, "amount"] = (legit.loc[noisy_idx, "avg_amount_30d"] * rng.lognormal(1.0, 0.5, n_noisy)).round(2)
    legit.loc[noisy_idx, "merchant_category"] = rng.choice(["viagem", "eletronicos", "restaurante"], n_noisy)

    # ---------- padrão NOVO de fraude a partir de 2026-06-01 ----------
    # "Ataque de aproximação em games digitais": muitos micro-pagamentos por aproximação,
    # cartão presente, perto de casa. Não existe antes de junho -> demonstra adaptação sem re-treino.
    n_new = int(n_rows * 0.006)
    new = build(n_new, fraud=False)  # base legítima, depois sobrescreve o padrão
    new["transaction_ts"] = pd.Timestamp("2026-06-01") + pd.to_timedelta(
        rng.uniform(0, (END - pd.Timestamp("2026-06-01")).total_seconds(), n_new), unit="s")
    new["merchant_category"] = "games_digitais"
    new["channel"] = "aproximacao"
    new["card_present"] = True
    new["amount"] = rng.uniform(19, 49, n_new).round(2)
    new["txn_count_24h"] = rng.poisson(9, n_new) + 4
    new["distinct_merchants_24h"] = 1
    new["distance_from_home_km"] = rng.exponential(3, n_new).round(1)
    new["is_international"] = False
    new["new_device"] = False
    new["is_fraud"] = True

    df = pd.concat([legit, fraud, new], ignore_index=True)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df.insert(0, "transaction_id", [f"TX{seed:02d}{i:08d}" for i in range(len(df))])

    # features derivadas (ficam ≤ 20 features no total)
    df["amount_to_avg_ratio"] = (df["amount"] / df["avg_amount_30d"]).round(3)
    df["hour_of_day"] = df["transaction_ts"].dt.hour.astype(int)
    df["day_of_week"] = df["transaction_ts"].dt.dayofweek.astype(int)
    df["transaction_ts"] = df["transaction_ts"].dt.floor("s")

    cols = ["transaction_id", "customer_id", "transaction_ts", "amount", "merchant_category", "channel",
            "card_present", "is_international", "distance_from_home_km", "new_device", "txn_count_24h",
            "distinct_merchants_24h", "failed_auth_24h", "customer_age", "account_age_days",
            "avg_amount_30d", "amount_to_avg_ratio", "credit_limit_utilization", "prior_chargebacks",
            "hour_of_day", "day_of_week", "is_fraud"]
    return df[cols]

In [ ]:
df = generate_transactions(N_ROWS, FRAUD_RATE, SEED)
print(f"{len(df):,} transações | fraude = {df.is_fraud.mean():.2%} | período {df.transaction_ts.min():%Y-%m-%d} → {df.transaction_ts.max():%Y-%m-%d}")
df.head()

In [ ]:
job = bq.load_table_from_dataframe(
    df, f"{FQ}.transactions",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result()
tbl = bq.get_table(f"{FQ}.transactions")
print(f"Carregado: {tbl.num_rows:,} linhas em {FQ}.transactions")
pd.DataFrame([(s.name, s.field_type) for s in tbl.schema], columns=["coluna", "tipo"]).T

### 2.1 Exploração rápida

Fraude por mês (repare no salto em junho, quando surge o padrão novo) e por canal.

In [ ]:
by_month = run_sql(f"""
SELECT FORMAT_TIMESTAMP('%Y-%m', transaction_ts) AS mes,
       COUNT(*) AS transacoes,
       COUNTIF(is_fraud) AS fraudes,
       ROUND(100 * COUNTIF(is_fraud) / COUNT(*), 2) AS pct_fraude
FROM {T}
GROUP BY mes ORDER BY mes
""", show=False)

by_channel = run_sql(f"""
SELECT channel, COUNT(*) AS transacoes,
       ROUND(100 * COUNTIF(is_fraud) / COUNT(*), 2) AS pct_fraude
FROM {T} GROUP BY channel ORDER BY pct_fraude DESC
""", show=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(by_month.mes, by_month.pct_fraude, color=BLUE, width=0.6)
ax[0].set_title("Taxa de fraude por mês (%)"); ax[0].tick_params(axis="x", rotation=45)
ax[1].barh(by_channel.channel, by_channel.pct_fraude, color=BLUE, height=0.6)
ax[1].set_title("Taxa de fraude por canal (%)"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
display(by_month)

## 3. Primeira previsão *zero-shot* com `AI.PREDICT`

Estratégia de validação honesta: **split temporal**. Treino = janeiro a junho; predição = julho (dados que o modelo nunca viu, com o rótulo real disponível para avaliar).

```sql
AI.PREDICT(
  { TABLE tabela_treino | (query_treino) },      -- exemplos rotulados (in-context)
  { TABLE tabela_predicao | (query_predicao) },  -- linhas a prever
  [, label_col => 'nome_do_rotulo' ]             -- default: 'label'
)
```

Regras que importam:
- **Toda coluna do treino que não é o rótulo vira feature.** Por isso tiramos IDs e timestamp com `EXCEPT`.
- A tabela de predição precisa ter **todas as features**; pode ter colunas extras (mantemos `transaction_id` e `is_fraud` para juntar e avaliar depois).
- Tipos suportados: `STRING`, `BOOL`, `INT64`, `FLOAT64`, `NUMERIC`, `BIGNUMERIC`. `TIMESTAMP` não; por isso derivamos `hour_of_day` e `day_of_week`.

Nenhum `CREATE MODEL`, nenhum endpoint, nenhum ajuste de hiperparâmetro. Só SQL.

In [ ]:
TRAIN_SQL = f"""
  SELECT * EXCEPT(transaction_id, customer_id, transaction_ts)
  FROM {T}
  WHERE transaction_ts < '2026-07-01'
"""
PREDICT_SQL = f"""
  SELECT *
  FROM {T}
  WHERE transaction_ts >= '2026-07-01'
"""

preview = run_sql(f"""
SELECT transaction_id, amount, merchant_category, channel,
       is_fraud,                                  -- rótulo real (só para conferência)
       predicted_is_fraud,                        -- previsão
       predicted_is_fraud_probs                   -- ARRAY<STRUCT<label STRING, prob FLOAT64>>
FROM AI.PREDICT(
  ({TRAIN_SQL}),
  ({PREDICT_SQL}),
  label_col => 'is_fraud')
LIMIT 20
""")

O resultado traz **todas as colunas da tabela de predição** mais:

- `predicted_is_fraud` (`BOOL`, mesmo tipo do rótulo)
- `predicted_is_fraud_probs`: `ARRAY<STRUCT<label STRING, prob FLOAT64>>` com a probabilidade de cada classe (`'true'`/`'false'`)

A probabilidade é o que interessa para o negócio: permite **calibrar o limiar** conforme a capacidade da mesa de análise, em vez de aceitar o corte padrão.

### 3.1 Materializar as previsões

Na prática você grava o resultado em uma tabela (ou agenda a query). Aqui extraímos a probabilidade de fraude para uma coluna escalar.

In [ ]:
_ = run_sql(f"""
CREATE OR REPLACE TABLE `{FQ}.predictions_tabfm` AS
SELECT
  transaction_id, transaction_ts, customer_id, amount, merchant_category, channel,
  is_fraud,
  predicted_is_fraud,
  (SELECT p.prob FROM UNNEST(predicted_is_fraud_probs) AS p WHERE p.label = 'true') AS fraud_prob
FROM AI.PREDICT(
  ({TRAIN_SQL}),
  ({PREDICT_SQL}),
  label_col => 'is_fraud')
""", show=False)

run_sql(f"""
SELECT COUNT(*) AS transacoes_julho,
       COUNTIF(is_fraud) AS fraudes_reais,
       COUNTIF(predicted_is_fraud) AS alertas_gerados,
       COUNTIF(is_fraud AND predicted_is_fraud) AS fraudes_capturadas
FROM `{FQ}.predictions_tabfm`
""");

## 4. Avaliação

### 4.1 `AI.EVALUATE`: métricas com uma linha

Mesma assinatura do `AI.PREDICT`, mas a segunda tabela precisa conter o rótulo real. Para classificação retorna `precision`, `recall`, `accuracy`, `f1_score`; para regressão `mean_absolute_error`, `r2_score`, etc.

In [ ]:
metrics_tabfm = run_sql(f"""
SELECT *
FROM AI.EVALUATE(
  ({TRAIN_SQL}),
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts) FROM {T} WHERE transaction_ts >= '2026-07-01'),
  label_col => 'is_fraud')
""")

### 4.2 Matriz de confusão e limiar de decisão

Em fraude, `accuracy` engana (99 % das transações são legítimas). O que importa é a troca entre **recall** (fraude capturada) e **precisão** (alertas que valem a pena). Como temos a probabilidade, calculamos as métricas para vários limiares direto em SQL.

In [ ]:
cm = run_sql(f"""
SELECT is_fraud AS real, predicted_is_fraud AS previsto, COUNT(*) AS n
FROM `{FQ}.predictions_tabfm`
GROUP BY 1, 2 ORDER BY 1 DESC, 2 DESC
""", show=False)
display(cm.pivot(index="real", columns="previsto", values="n").fillna(0).astype(int))

thr = run_sql(f"""
WITH t AS (SELECT threshold FROM UNNEST([0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]) AS threshold)
SELECT threshold,
       COUNTIF(fraud_prob >= threshold) AS alertas,
       ROUND(COUNTIF(is_fraud AND fraud_prob >= threshold) / COUNTIF(fraud_prob >= threshold), 3) AS precisao,
       ROUND(COUNTIF(is_fraud AND fraud_prob >= threshold) / COUNTIF(is_fraud), 3) AS recall,
       ROUND(SUM(IF(is_fraud AND fraud_prob >= threshold, amount, 0)), 2) AS valor_fraude_bloqueado
FROM `{FQ}.predictions_tabfm`, t
GROUP BY threshold ORDER BY threshold
""")

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

pred = run_sql(f"SELECT is_fraud, fraud_prob, amount, transaction_ts FROM `{FQ}.predictions_tabfm`", show=False)
auc = roc_auc_score(pred.is_fraud, pred.fraud_prob)
ap  = average_precision_score(pred.is_fraud, pred.fraud_prob)
print(f"AUC-ROC = {auc:.4f} | AUC-PR (average precision) = {ap:.4f}")

p, r, _ = precision_recall_curve(pred.is_fraud, pred.fraud_prob)
fig, ax = plt.subplots(figsize=(5.2, 4))
ax.plot(r, p, color=BLUE, lw=2)
ax.axhline(pred.is_fraud.mean(), color=GRAY, ls="--", lw=1, label=f"baseline aleatório ({pred.is_fraud.mean():.1%})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precisão"); ax.set_title("Curva precision-recall (TabFM, julho/2026)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

### 4.3 Visão de operação: capacidade da mesa de análise

Pergunta típica do negócio: *"minha equipe consegue revisar 40 alertas por dia. Quantas fraudes eu pego priorizando pela probabilidade?"*

In [ ]:
ALERTAS_POR_DIA = 40

run_sql(f"""
WITH ranked AS (
  SELECT DATE(transaction_ts) AS dia, is_fraud, amount, fraud_prob,
         ROW_NUMBER() OVER (PARTITION BY DATE(transaction_ts) ORDER BY fraud_prob DESC) AS rk
  FROM `{FQ}.predictions_tabfm`
)
SELECT
  {ALERTAS_POR_DIA} AS alertas_por_dia,
  COUNTIF(rk <= {ALERTAS_POR_DIA}) AS alertas_revisados,
  COUNTIF(rk <= {ALERTAS_POR_DIA} AND is_fraud) AS fraudes_encontradas,
  ROUND(COUNTIF(rk <= {ALERTAS_POR_DIA} AND is_fraud) / COUNTIF(rk <= {ALERTAS_POR_DIA}), 3) AS precisao_da_fila,
  ROUND(COUNTIF(rk <= {ALERTAS_POR_DIA} AND is_fraud) / COUNTIF(is_fraud), 3) AS recall_total,
  ROUND(SUM(IF(rk <= {ALERTAS_POR_DIA} AND is_fraud, amount, 0)), 2) AS valor_recuperado,
  ROUND(SUM(IF(is_fraud, amount, 0)), 2) AS valor_total_fraude
FROM ranked
""");

## 5. Adaptação a novos padrões sem re-treino

Aqui está a mudança de paradigma. Com um modelo treinado, quando surge um golpe novo você precisa **coletar rótulos, re-treinar, validar e re-implantar**. Com TabFM, o "modelo" **é a query**: basta a janela de treino incluir os exemplos recentes.

Nosso dataset tem um padrão que só aparece a partir de junho (`games_digitais` + `aproximacao`, micro-pagamentos repetidos). Vamos comparar duas execuções sobre o **mesmo julho**:

- **Janela desatualizada:** treino jan-mai (o padrão novo não existe no contexto)
- **Janela atual:** treino jan-jun (o padrão novo já apareceu)

In [ ]:
_ = run_sql(f"""
CREATE OR REPLACE TABLE `{FQ}.predictions_tabfm_stale` AS
SELECT transaction_id, is_fraud, predicted_is_fraud,
       (SELECT p.prob FROM UNNEST(predicted_is_fraud_probs) AS p WHERE p.label = 'true') AS fraud_prob
FROM AI.PREDICT(
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts)
   FROM {T} WHERE transaction_ts < '2026-06-01'),          -- <- única diferença
  ({PREDICT_SQL}),
  label_col => 'is_fraud')
""", show=False)

drift = run_sql(f"""
WITH j AS (
  SELECT t.merchant_category, t.channel, t.is_fraud,
         s.predicted_is_fraud AS pred_stale, f.predicted_is_fraud AS pred_fresh
  FROM {T} t
  JOIN `{FQ}.predictions_tabfm_stale` s USING (transaction_id)
  JOIN `{FQ}.predictions_tabfm`       f USING (transaction_id)
)
SELECT IF(merchant_category = 'games_digitais' AND channel = 'aproximacao', 'padrao_novo', 'padroes_conhecidos') AS grupo,
       COUNTIF(is_fraud) AS fraudes,
       ROUND(COUNTIF(is_fraud AND pred_stale) / COUNTIF(is_fraud), 3) AS recall_treino_jan_mai,
       ROUND(COUNTIF(is_fraud AND pred_fresh) / COUNTIF(is_fraud), 3) AS recall_treino_jan_jun
FROM j GROUP BY grupo ORDER BY grupo
""")

fig, ax = plt.subplots(figsize=(6, 3.4))
x = np.arange(len(drift)); w = 0.36
ax.bar(x - w/2, drift.recall_treino_jan_mai, w, color=GRAY,  label="treino jan-mai (desatualizado)")
ax.bar(x + w/2, drift.recall_treino_jan_jun, w, color=BLUE,  label="treino jan-jun (atual)")
ax.set_xticks(x); ax.set_xticklabels(drift.grupo); ax.set_ylim(0, 1.05)
ax.set_title("Recall em julho por grupo de fraude"); ax.legend(frameon=False, fontsize=8)
for i, (a, b) in enumerate(zip(drift.recall_treino_jan_mai, drift.recall_treino_jan_jun)):
    ax.text(i - w/2, a + 0.02, f"{a:.0%}", ha="center", fontsize=8); ax.text(i + w/2, b + 0.02, f"{b:.0%}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

Sem nenhuma etapa de re-treino, o padrão novo passa a ser capturado assim que entra no histórico. Em produção isso vira uma **view ou query agendada com janela deslizante** (por exemplo, `WHERE transaction_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 180 DAY)`).

## 6. Comparação com BigQuery ML clássico (`BOOSTED_TREE_CLASSIFIER`)

O TabFM **complementa** os modelos treinados do BigQuery ML; não os substitui. Para que o cliente escolha bem, vamos rodar o fluxo tradicional sobre os mesmos dados e comparar.

In [ ]:
t0 = time.time()
_ = run_sql(f"""
CREATE OR REPLACE MODEL `{FQ}.fraud_boosted_tree`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['is_fraud'],
  max_iterations = 50,
  early_stop = TRUE,
  data_split_method = 'NO_SPLIT'
) AS
{TRAIN_SQL}
""", show=False)
train_time_bt = time.time() - t0
print(f"Treino do BOOSTED_TREE: {train_time_bt/60:.1f} min")

metrics_bt = run_sql(f"""
SELECT precision, recall, accuracy, f1_score, roc_auc
FROM ML.EVALUATE(MODEL `{FQ}.fraud_boosted_tree`,
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts) FROM {T} WHERE transaction_ts >= '2026-07-01'))
""")

In [ ]:
_ = run_sql(f"""
CREATE OR REPLACE TABLE `{FQ}.predictions_bt` AS
SELECT transaction_id, is_fraud, predicted_is_fraud,
       (SELECT p.prob FROM UNNEST(predicted_is_fraud_probs) AS p WHERE CAST(p.label AS STRING) = 'true') AS fraud_prob
FROM ML.PREDICT(MODEL `{FQ}.fraud_boosted_tree`, ({PREDICT_SQL}))
""", show=False)

pred_bt = run_sql(f"SELECT is_fraud, fraud_prob FROM `{FQ}.predictions_bt`", show=False)
comp = pd.DataFrame({
    "TabFM (AI.PREDICT)": [auc, ap, metrics_tabfm.precision[0], metrics_tabfm.recall[0], metrics_tabfm.f1_score[0]],
    "BOOSTED_TREE (CREATE MODEL)": [roc_auc_score(pred_bt.is_fraud, pred_bt.fraud_prob),
                                    average_precision_score(pred_bt.is_fraud, pred_bt.fraud_prob),
                                    metrics_bt.precision[0], metrics_bt.recall[0], metrics_bt.f1_score[0]],
}, index=["AUC-ROC", "AUC-PR", "precision", "recall", "f1_score"]).round(4)
display(comp)

Os dois chegam a qualidade parecida neste problema (é o esperado: dados tabulares bem comportados). Na execução de referência deste notebook:

| | TabFM (`AI.PREDICT`) | `BOOSTED_TREE_CLASSIFIER` |
|---|---|---|
| precision / recall / F1 | 0,988 / 0,824 / 0,899 | 0,971 / 0,830 / 0,895 |
| AUC-ROC / AUC-PR | 0,985 / 0,928 | 0,985 / 0,925 |
| tempo até a primeira previsão | ≈ 4 min (uma query) | ≈ 20 min de treino + `ML.PREDICT` |

A diferença está no **caminho** até o resultado:

| | TabFM (`AI.PREDICT`) | BQML treinado (`BOOSTED_TREE`, `LOGISTIC_REG`, `AUTOML`...) |
|---|---|---|
| Passos | 1 query | `CREATE MODEL` → `ML.EVALUATE` → `ML.PREDICT` (+ re-treino periódico) |
| Artefato para gerenciar | nenhum | modelo versionado no dataset |
| Hiperparâmetros | nenhum | `max_iterations`, `learn_rate`, profundidade, regularização... |
| Novos padrões | mudar a janela do `WHERE` | re-treinar e re-validar |
| Explicabilidade | não há importância de features | `ML.EXPLAIN_PREDICT`, `ML.FEATURE_IMPORTANCE` |
| Features | até 20 colunas (mais sob consulta) | centenas |
| Classes | até 10 | sem limite prático |
| Volume de treino | pequeno a médio (amostragem inteligente) | dezenas de milhões de linhas |
| Latência de inferência | minutos por lote; sem endpoint online | lote via SQL; ou exportar para Vertex AI online |

**Escolha TabFM quando:** precisa de previsão rápida e de alta qualidade sem time de ML; histórico pequeno/médio; dados mudam com frequência; casos conversacionais/agentes (BigQuery MCP server) que pedem previsão sob demanda.

**Escolha modelo treinado quando:** volume de treino muito grande; precisa controlar hiperparâmetros; mais de 20 features; precisa de importância de features/explicabilidade regulatória; precisa de inferência online de baixa latência.

## 7. Regressão com a mesma sintaxe

Basta o rótulo ser numérico. Como exemplo didático, prevemos o **valor da transação** a partir das demais colunas (em um caso real: valor esperado de perda, LTV, limite recomendado, tempo até chargeback...). O `AI.EVALUATE` passa a devolver métricas de regressão.

In [ ]:
run_sql(f"""
SELECT *
FROM AI.EVALUATE(
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts, is_fraud, amount_to_avg_ratio)
   FROM {T} WHERE transaction_ts < '2026-07-01'),
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts, is_fraud, amount_to_avg_ratio)
   FROM {T} WHERE transaction_ts >= '2026-07-01'),
  label_col => 'amount')
""");

> Retiramos `amount_to_avg_ratio` porque ela é derivada do próprio rótulo (vazamento). Boa prática que vale para qualquer modelo.

## 8. Operacionalização

**Query agendada (scoring diário) com janela deslizante de treino:**

```sql
-- Scheduled query: roda todo dia às 06:00
CREATE OR REPLACE TABLE `PROJETO.DATASET.fraud_scores_hoje` AS
SELECT transaction_id, transaction_ts, customer_id, amount,
       predicted_is_fraud,
       (SELECT p.prob FROM UNNEST(predicted_is_fraud_probs) p WHERE p.label = 'true') AS fraud_prob
FROM AI.PREDICT(
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts)
   FROM `PROJETO.DATASET.transactions_rotuladas`
   WHERE transaction_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 180 DAY)),
  (SELECT * FROM `PROJETO.DATASET.transactions_novas`
   WHERE DATE(transaction_ts) = CURRENT_DATE() - 1),
  label_col => 'is_fraud');
```

**Encapsular em uma *table function* para analistas e agentes:**

```sql
CREATE OR REPLACE TABLE FUNCTION `PROJETO.DATASET.score_fraude`(inicio TIMESTAMP, fim TIMESTAMP) AS
SELECT * FROM AI.PREDICT(
  (SELECT * EXCEPT(transaction_id, customer_id, transaction_ts)
   FROM `PROJETO.DATASET.transactions_rotuladas` WHERE transaction_ts < inicio),
  (SELECT * FROM `PROJETO.DATASET.transactions` WHERE transaction_ts BETWEEN inicio AND fim),
  label_col => 'is_fraud');
-- SELECT * FROM `PROJETO.DATASET.score_fraude`('2026-07-01', '2026-07-31');
```

**Agentes:** via *BigQuery MCP server*, um agente (Gemini, ADK, Claude, etc.) pode chamar `AI.PREDICT` como ferramenta: "quais clientes têm maior risco de churn este mês?" vira uma query, sem runtime de ML para operar.

**Custo (Preview):** cobrado em *slots* (Enterprise/Enterprise Plus) ou por *bytes processados* (on-demand). A partir de **30/10/2026** passa a cobrança por **tokens do TabFM** + slots/bytes do restante da query. Verifique os números da célula `run_sql` para o volume desta demo.

## 9. Limitações atuais (Preview)

- Até **20 colunas de feature** (mais: solicitar via `bqml-feedback@google.com`).
- Classificação em até **10 classes**.
- Tipos: `STRING`, `BOOL`, `INT64`, `FLOAT64`, `NUMERIC`, `BIGNUMERIC` (sem `TIMESTAMP`, `ARRAY`, `STRUCT`, `GEOGRAPHY`; derive features antes).
- Sem importância de features nativa.
- Treino grande é amostrado automaticamente; para dezenas de milhões de linhas de treino prefira modelo treinado.
- Inferência em lote (minutos), não online.

## 10. Resumo

| O que o cliente viu | Como |
|---|---|
| Previsão de fraude sem treinar modelo | 1 `SELECT ... FROM AI.PREDICT(...)` |
| Probabilidades para calibrar alertas | `predicted_is_fraud_probs` + SQL |
| Métricas oficiais | `AI.EVALUATE` |
| Adaptação a golpe novo sem re-treino | mudar a janela de treino |
| Quando usar TabFM × modelo treinado | comparação lado a lado com `BOOSTED_TREE` |
| Regressão | mesma função, rótulo numérico |

In [ ]:
# Limpeza (opcional): remove tudo que a demo criou
# bq.delete_dataset(FQ, delete_contents=True, not_found_ok=True); print("Dataset removido:", FQ)